In [1]:
import os
import warnings
import folium
import rasterio

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import nivapy3 as nivapy
# import correct_catchments as cc

from shapely.geometry import Polygon

plt.style.use("ggplot")
warnings.simplefilter("ignore")

In [2]:
eng = nivapy.da.connect_postgis()

Connection successful.


# Delineate catchment boundaries for water chem monitoring points

Delineate catchment boundaries for each river monitoring point from combined vannmiljø and quantom organic carbon monitoring for TOC and DOC parameters.

# User input

In [3]:
# User input

# Delineation settings

# Use a constant minimum area threshold? If not, assume it is specified in the df of outlet points
constant_min_threshold = False  # True or False

# If constant, specify the smallest catchments for delineation.
min_size_km2 = 0.2

# monitoring point id column (in the point input file)
id_col = 'site_code'

# Dem resolution (10, 20 or 40 m)
dem_res = 20  # metres

# Directory to read raw shapefile from and to write processed shp to
cur_dir = r"/home/jovyan/shared/common/norsink/spatial/monitored_chem_catchments"

# filename with lat/lon points to delineate
# fname = "outlet_locations_vm-rivs-quant-oc_b4-delineation.csv"
in_pt_fname = "outlet_locations_vm-rivs-quant-oc_v3-for-redelineation.csv"

# filename for saving delineated catchment boundaries
# out_cb_fname = f"raw_catchments_vm-quant_{dem_res}-dem.shp"
out_cb_fname = f"raw_catchments_redelineation_v3_{dem_res}-dem.shp"

# Read in data

In [4]:
# Read in outlets
outlet_df = pd.read_csv(os.path.join(cur_dir, in_pt_fname))
print(len(outlet_df), "River stations in Norway for delineation")

# if 'geometry' is in the cols, drop (as it's a csv...)
outlet_df = outlet_df.drop(columns='geometry', errors='ignore')

outlet_df

16 River stations in Norway for delineation


,site_code,lon,lat,site_id,Name,start_year,end_year,n_years,n_samples,av_freq,area_km2_v1,is_lake_raw_v1,dist_to_cb_v1,inside_v1,outside_v1,f_outside_no_v1,min_area_km2,delineate_v2,drop_other,is_lake
0,002-50935,11.408067,59.368878,50935,Rakkestad kommune St 4,2008,2024,16.1,15,0.93,3.820415,False,9.009502,False,False,-0.000206,5.0,True,False,False
1,002-59171,11.150695,59.561493,59171,Engerbekken oppstrøms renseanlegg,2011,2025,14.1,11,0.78,5.600887,False,3.333703,False,False,-0.000341,5.0,True,False,False
2,002-87939,11.148889,59.560910,87939,Engerbekken etter utløp renseanlegg,2011,2017,5.9,7,1.18,11.365195,False,0.758003,False,False,-0.000352,1.0,False,False,False
3,006-56474,10.710520,59.937720,56474,"Frognerelva, Sognsvannsbekken",2007,2022,15.1,20,1.32,4.052007,False,5.407403,False,False,-0.000591,0.5,False,False,False
4,006-56475,10.711020,59.937280,56475,Gaustadbekken,2007,2022,15.1,20,1.32,4.052007,False,5.010297,False,False,-0.000591,0.5,False,False,False
5,012-118926,9.237260,60.732400,118926,Liaåni bekkefelt nedstrøms Kyrkja renseanlegg,2024,2025,1.4,17,12.30,2.030354,False,4.415504,False,False,-0.001599,0.2,True,False,False
6,012-65200,10.172151,59.747904,65200,Drammenselva nedstrøms Muusøya avløpsanlegg,2014,2025,11.5,78,6.77,0.670929,False,5.901597,False,False,-0.001001,1000.0,False,False,False
7,015-53043,9.127590,60.006880,53043,Sagstugobekken,2008,2008,1.0,5,5.00,7.363364,False,7.332777,False,False,-0.001825,1.0,False,False,False
8,015-65238,9.125980,60.006970,65238,Medåe 2,2008,2008,1.0,5,5.00,7.363364,False,10.276466,False,False,-0.001825,1.0,False,False,False
9,016-114814,8.132800,59.744803,114814,Bjønnskarhaugen utløp Farvhovdtjønn,2023,2024,1.2,10,8.03,0.224562,False,7.532025,False,False,-0.002839,5.0,True,False,False


In [5]:
# test_df = outlet_df.sample(n=5, random_state=24)
# test_df

In [6]:
# Make geodataframe of outlets
stn_gdf = gpd.GeoDataFrame(outlet_df, geometry=gpd.points_from_xy(outlet_df.lon, outlet_df.lat), crs="EPSG:4326")
stn_gdf = stn_gdf.to_crs("EPSG:25833")
stn_gdf = stn_gdf.rename({'lon': 'longitude', 'lat': 'latitude'}, axis=1).reset_index()

# Interactive map of outlets
nivapy.spatial.quickmap(
    stn_gdf, kartverket=True, popup=id_col, aerial_imagery=False
)

# Delineate catchments

In [7]:
# Catchment boundaries and areas

if constant_min_threshold == True:
    raw_cat_gdf = nivapy.spatial.derive_watershed_boundaries(
        outlet_df,
        id_col=id_col,
        xcol="lon",
        ycol="lat",
        crs=f"epsg:{4326}",
        min_size_km2=min_size_km2,
        dem_res_m=dem_res,
        reproject=False,  # keep in EPSG 25833
    )

else:

    if 'min_area_km2' not in outlet_df.columns:
        raise ValueError("Missing 'min_area_km2' column")

    n_groups = outlet_df['min_area_km2'].nunique()

    cat_gdf_li = []

    for i, (min_area, grp_df) in enumerate(outlet_df.groupby('min_area_km2'), start=1):

        print("")
        print(f"Processing group {i}/{n_groups} (min_area={min_area})")
        print("")

        cat_gdf_grp = nivapy.spatial.derive_watershed_boundaries(
            grp_df,
            id_col=id_col,
            xcol="lon",
            ycol="lat",
            crs="epsg:4326",
            min_size_km2=min_area,
            dem_res_m=dem_res,
            reproject=False,
        )

        cat_gdf_li.append(cat_gdf_grp)

        # This was indented originally. Why?! Wrong results?!
    raw_cat_gdf = pd.concat(cat_gdf_li, axis=0, ignore_index=True)

# Compute area
raw_cat_gdf["area_km2"] = raw_cat_gdf.to_crs({"proj": "cea"})["geometry"].area / 1e6

raw_cat_gdf.head()


Processing group 1/7 (min_area=0.2)

Connection successful.


Looping over vassdragsområder:   0%|          | 0/4 [00:00<?, ?it/s]

Looping over outlets in vassdragsområder 012:   0%|          | 0/1 [00:00<?, ?it/s]

Looping over outlets in vassdragsområder 067:   0%|          | 0/1 [00:00<?, ?it/s]

Looping over outlets in vassdragsområder 078:   0%|          | 0/1 [00:00<?, ?it/s]

...

Looping over outlets in vassdragsområder 024:   0%|          | 0/1 [00:00<?, ?it/s]


Processing group 7/7 (min_area=1000.0)

Connection successful.


Looping over vassdragsområder:   0%|          | 0/1 [00:00<?, ?it/s]

Looping over outlets in vassdragsområder 012:   0%|          | 0/1 [00:00<?, ?it/s]

,site_code,geometry,lon,lat,site_id,Name,start_year,end_year,n_years,n_samples,...,is_lake_raw_v1,dist_to_cb_v1,inside_v1,outside_v1,f_outside_no_v1,min_area_km2,delineate_v2,drop_other,is_lake,area_km2
0,012-118926,"POLYGON ((186160 6746800, 186160 6746780, 1861...",9.237260,60.73240,118926,Liaåni bekkefelt nedstrøms Kyrkja renseanlegg,2024,2025,1.4,17,...,False,4.415504,False,False,-0.001599,0.2,True,False,False,0.600627
1,067-41892,"POLYGON ((-11100 6781840, -11100 6781820, -111...",5.568370,60.83091,41892,"Svartetjørna, utløp",1994,2023,29.4,796,...,False,19.419895,False,False,-0.005625,0.2,True,False,False,0.603008
2,078-63938,"POLYGON ((61440 6838740, 61440 6838720, 61360 ...",6.758498,61.42752,63938,Tverrgrovi (Fjærland),2014,2024,10.2,20,...,False,46.537894,True,False,-0.003925,0.2,True,False,False,0.511591
3,123-102835,"POLYGON ((280400 7040120, 280400 7040100, 2803...",10.617000,63.42006,102835,Øyåsbrubekken oppstrøms E6,2021,2025,4.6,18,...,False,1.128379,False,False,-0.000380,0.2,True,False,False,0.558188
4,006-56474,"POLYGON ((260540 6659980, 260540 6659960, 2605...",10.710520,59.93772,56474,"Frognerelva, Sognsvannsbekken",2007,2022,15.1,20,...,False,5.407403,False,False,-0.000591,0.5,False,False,False,12.975423


# Drop multiplygons

In [8]:
# How many multipolygons are there?
multipolygon_rows = raw_cat_gdf['geometry'].apply(lambda geom: geom.geom_type == 'MultiPolygon')
print(f"{multipolygon_rows.sum()} out of {len(raw_cat_gdf)} catchment boundaries have MultiPolygon geometry")

# Pick only the largest polygon per multipolygon based on area
scs_cleaned_li = []
for idx, row in raw_cat_gdf.iterrows():
    if row.geometry.geom_type == 'MultiPolygon':
        # Find the largest polygon based on area
        main_polygon = max(row.geometry.geoms, key=lambda polygon: polygon.area)
        # Create a new row with the main polygon
        new_row = row.copy()
        new_row.geometry = Polygon(main_polygon)
        scs_cleaned_li.append(new_row)
    else:
        # If it's not a MultiPolygon, just append the row as is
        scs_cleaned_li.append(row)

drop_multipolygons_df = pd.DataFrame(scs_cleaned_li)
cat_gdf = gpd.GeoDataFrame(drop_multipolygons_df, geometry='geometry', crs=raw_cat_gdf.crs)
cat_gdf.tail()

2 out of 16 catchment boundaries have MultiPolygon geometry


,site_code,geometry,lon,lat,site_id,Name,start_year,end_year,n_years,n_samples,...,is_lake_raw_v1,dist_to_cb_v1,inside_v1,outside_v1,f_outside_no_v1,min_area_km2,delineate_v2,drop_other,is_lake,area_km2
11,002-59171,"POLYGON ((281700 6612440, 281700 6612420, 2816...",11.150695,59.561493,59171,Engerbekken oppstrøms renseanlegg,2011,2025,14.1,11,...,False,3.333703,False,False,-0.000341,5.0,True,False,False,11.365195
12,016-114814,"POLYGON ((115960 6645380, 115960 6645360, 1159...",8.132800,59.744803,114814,Bjønnskarhaugen utløp Farvhovdtjønn,2023,2024,1.2,10,...,False,7.532025,False,False,-0.002839,5.0,True,False,False,8.990270
13,083-82201,"POLYGON ((8500 6845360, 8500 6845340, 8480 684...",5.673860,61.379540,82201,Kvamselv oppstraums samløp Vassenden,2016,2021,5.1,13,...,False,6.518506,False,False,-0.005241,10.0,True,False,False,28.382768
14,024-98549,"POLYGON ((49240 6520280, 49240 6520260, 49220 ...",7.113794,58.192940,98549,Lygna ved Dyrøya,2020,2025,5.1,47,...,False,21.823132,False,False,-0.004455,100.0,True,False,False,478.484065
15,012-65200,"POLYGON ((176380 6708620, 176380 6708760, 1763...",10.172151,59.747904,65200,Drammenselva nedstrøms Muusøya avløpsanlegg,2014,2025,11.5,78,...,False,5.901597,False,False,-0.001001,1000.0,False,False,False,17057.393210


In [9]:
# Save shapefiles with catchments for all rivers
cat_gdf.to_file(os.path.join(cur_dir, "shp", out_cb_fname))

In [10]:
cat_gdf.describe()

,lon,lat,site_id,start_year,end_year,n_years,n_samples,av_freq,area_km2_v1,dist_to_cb_v1,f_outside_no_v1,min_area_km2,area_km2
count,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000
mean,9.051080,60.127529,76302.437500,2012.125000,2021.375000,9.562500,79.187500,6.170625,3.222753,14.871973,-0.002122,70.675000,1103.150990
std,1.963135,1.267194,24819.203283,7.632169,5.619905,7.863237,196.355195,6.754324,3.288681,21.481406,0.001856,249.032845,4256.088409
min,5.568370,58.184261,41892.000000,1994.000000,2008.000000,1.000000,5.000000,0.780000,0.224562,0.758003,-0.005625,0.200000,0.511591
25%,7.878049,59.561347,56474.750000,2008.000000,2021.750000,3.800000,10.750000,1.320000,0.622351,4.861599,-0.003352,0.425000,1.121170
50%,9.182425,59.937500,65219.000000,2011.000000,2024.000000,8.050000,17.500000,4.440000,2.198973,6.925642,-0.001712,1.000000,8.490946
75%,10.710645,60.757027,99620.500000,2017.000000,2025.000000,15.100000,26.750000,8.317500,4.439227,12.562323,-0.000538,5.000000,14.341723
max,11.408067,63.420060,118926.000000,2024.000000,2025.000000,29.400000,796.000000,27.110000,11.365195,83.546489,-0.000206,1000.000000,17057.393210
